# 03 - My one-move study setup

I freeze 1,000 fresh **one-solver-move puzzles** for assisted solving. My first 100 training cases are the shared early-results pilot. I keep 200 held-out puzzles untouched. This notebook makes no API calls. I rerun it only to verify the same frozen study, not to resample.

In [1]:
RUN_MODE = "setup"
WAIT_FOR_QUOTA = False


In [2]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
import os, json, hashlib, re, time, math, uuid, warnings, platform
import requests, chess, nbformat
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'experiments/stage_index.json').exists())
CFG = ROOT/'experiments/one_move_v1'
OUT = ROOT/'data/experiments/one_move_v1'; OUT.mkdir(exist_ok=True)
CONFIG = json.loads((CFG/'config.json').read_text())
PLAN = json.loads((CFG/'analysis_plan.json').read_text())
PROMPT = (CFG/'prompt.txt').read_text()
assert RUN_MODE in ['setup', 'pilot', 'bulk'] and isinstance(WAIT_FOR_QUOTA, bool)
PACIFIC = ZoneInfo('America/Los_Angeles')
SECRETS = []  # I load credentials only if I explicitly enable live collection.

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def redact(value):
    if isinstance(value, str):
        for secret in SECRETS: value = value.replace(secret, '[REDACTED]')
        return value
    if isinstance(value, dict): return {k: redact(v) for k, v in value.items()}
    if isinstance(value, list): return [redact(v) for v in value]
    return value

def save(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(redact(value), indent=2, ensure_ascii=False)+'\n', encoding='utf-8')
    os.replace(temporary, path)

def records(path):
    path = Path(path)
    if not path.exists(): return []
    text = path.read_text(encoding='utf-8'); lines = text.splitlines(); rows = []
    for i, line in enumerate(lines):
        if not line.strip(): continue
        try: rows.append(json.loads(line))
        except json.JSONDecodeError:
            if i == len(lines)-1 and not text.endswith('\n'):
                warnings.warn(f'I found an incomplete final log record in {path.name}; it is not a completed response.')
            else: raise
    return rows

def append(path, row):
    path = Path(path)
    # I repair only an incomplete last line after a crash, under the provider file lock.
    if path.exists():
        raw = path.read_bytes()
        if raw and not raw.endswith(b'\n'):
            try: json.loads(raw.splitlines()[-1]); prefix = raw+b'\n'
            except json.JSONDecodeError: prefix = raw[:raw.rfind(b'\n')+1]
            path.write_bytes(prefix)
    with path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(redact(row), ensure_ascii=False)+'\n'); stream.flush(); os.fsync(stream.fileno())

@contextmanager
def provider_lock(folder):
    # OS locks are released if the kernel exits, so stale PID files cannot block a resume.
    with (folder/'collection.lock').open('a+b') as handle:
        if handle.tell() == 0: handle.write(b'0'); handle.flush()
        handle.seek(0)
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        try: yield
        finally:
            handle.seek(0)
            if os.name == 'nt': msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
            else: fcntl.flock(handle.fileno(), fcntl.LOCK_UN)

def code_hash(name):
    n = nbformat.read(ROOT/'notebooks'/name, as_version=4)
    code = '\n\n'.join(c.source for c in n.cells if c.cell_type == 'code' and 'run_control' not in c.metadata.get('tags', []))
    return hashlib.sha256(code.encode()).hexdigest()

manifest = {'version': CONFIG['version'], 'config_sha256': sha(CFG/'config.json'),
            'plan_sha256': sha(CFG/'analysis_plan.json'), 'final_plan_sha256': sha(CFG/'final_analysis_plan.json'), 'prompt_sha256': sha(CFG/'prompt.txt'),
            'collection_code_sha256': code_hash('04_collect_answers.ipynb'),
            'analysis_code_sha256': code_hash('06_final_analysis.ipynb'),
            'preparation_code_sha256': code_hash('03_prepare_study.ipynb'),
            'early_code_sha256': code_hash('05_early_results.ipynb')}
if (OUT/'manifest.json').exists():
    assert manifest == json.loads((OUT/'manifest.json').read_text()), 'Frozen study changed; create a separate version.'
else: save(OUT/'manifest.json', manifest)

# I protect historical raw responses and fitted models. Viewer-facing Markdown is editable.
if not (OUT/'historical_hashes.json').exists():
    protected = {p.relative_to(ROOT).as_posix(): sha(p)
                 for parent in ['data/experiments', 'data/analysis']
                 for p in (ROOT/parent).rglob('*') if p.is_file() and OUT not in p.parents}
    save(OUT/'historical_hashes.json', protected)
else: protected = json.loads((OUT/'historical_hashes.json').read_text())
assert all(sha(ROOT/p) == digest for p, digest in protected.items())
print('I verified my frozen protocol and protected earlier response/model files.')
def score(board,task,response,expected=None):
    choice=response['choices'][0]
    if choice.get('finish_reason')=='length':return 'generation_limit',None,False
    if choice.get('finish_reason')!='stop':return 'nonstop_termination',None,False
    answer=choice.get('message',{}).get('content')
    if not isinstance(answer,str):return 'unparseable',None,False
    answer=answer.strip()
    if task=='side':
        if answer not in ['white','black']:return 'unparseable',answer,False
        return ('success' if answer==('white' if board.turn else 'black') else 'incorrect_side'),answer,True
    if re.fullmatch('[a-h][1-8][a-h][1-8][qrbn]?',answer):
        move=chess.Move.from_uci(answer)
        if move not in board.legal_moves:return 'illegal_move',answer,False
    else:
        try:
            move=board.parse_san(answer)
            if move not in board.legal_moves:return 'illegal_move',answer,False
            canonical=board.san(move)
            normalized=answer.replace('0-0-0','O-O-O').replace('0-0','O-O')
            if normalized not in {canonical,canonical.rstrip('+#')}:return 'unparseable',answer,False
        except ValueError:return 'unparseable',answer,False
    answer=move.uci()
    if task=='mate':
        copy=board.copy();copy.push(move)
        correct=copy.is_checkmate()
    else:correct=task=='legal' or answer==expected
    return ('success' if correct else 'legal_incorrect'),answer,True
def seconds(value):
    try:return float(value)
    except (ValueError,TypeError):
        pairs=re.findall(r'([0-9.]+)(ms|s|m|h)',str(value))
        if pairs:return sum(float(n)*{'ms':.001,'s':1,'m':60,'h':3600}[unit] for n,unit in pairs)
        try:return max(0,(datetime.strptime(value,'%a, %d %b %Y %H:%M:%S GMT').replace(tzinfo=timezone.utc)-datetime.now(timezone.utc)).total_seconds())
        except (ValueError,TypeError):return None

def prompt_for(board, task='evaluation'):
    pieces = '\n'.join(('White' if color else 'Black')+' pieces: '+', '.join(
        f'{chess.piece_name(p.piece_type)} {chess.square_name(s)}'
        for s, p in sorted(board.piece_map().items()) if p.color == color)
        for color in [chess.WHITE, chess.BLACK])
    text = PROMPT.format(side='White' if board.turn else 'Black', fen=board.fen(),
                         pieces=pieces, choices=', '.join(sorted(m.uci() for m in board.legal_moves)))
    if task == 'side': return f'Which side moves next in this FEN? {board.fen()}\n'+pieces+'\nReturn only white or black in the final answer.'
    if task == 'mate': return text.replace('Choose the best next move', 'Choose a move that delivers checkmate immediately')
    if task == 'legal': return text.replace('Choose the best next move', 'Choose any legal next move')
    return text

def request_body(spec, board, task='evaluation'):
    text = prompt_for(board, task)
    if spec['provider'] == 'groq':
        return {'model': spec['model'], 'temperature': spec['temperature'], 'seed': spec['seed'],
                'reasoning_effort': spec['reasoning_effort'], 'max_completion_tokens': spec['max_completion_tokens'],
                'messages': [{'role': 'user', 'content': text}]}
    return {'contents': [{'role': 'user', 'parts': [{'text': text}]}],
            'generationConfig': {'temperature': spec['temperature'], 'topP': spec['top_p'], 'seed': spec['seed'],
                'maxOutputTokens': spec['max_completion_tokens'],
                'thinkingConfig': {'thinkingLevel': spec['thinking_level'], 'includeThoughts': False}}}

def normalized(spec, raw):
    if spec['provider'] == 'groq': return raw
    candidates = raw.get('candidates', [])
    if not candidates: return {'choices': [{'message': {'content': ''}, 'finish_reason': 'blocked_or_empty'}]}
    candidate = candidates[0]; reason = candidate.get('finishReason', 'UNKNOWN')
    content = ''.join(p.get('text', '') for p in candidate.get('content', {}).get('parts', []) if not p.get('thought', False))
    return {'choices': [{'message': {'content': content}, 'finish_reason': {'STOP': 'stop', 'MAX_TOKENS': 'length'}.get(reason, reason.lower())}]}

start = chess.Board(); after = start.copy(); after.push_uci('e2e4')
mate = chess.Board('7k/8/5KQ1/8/8/8/8/8 w - - 0 1'); mirror = mate.transform(chess.flip_horizontal)
FIXTURES = [('mate_a_white', mate, 'mate'), ('side_white', start, 'side'), ('legal_white', start, 'legal'),
            ('side_black', after, 'side'), ('legal_black', after, 'legal'), ('mate_a_black', mate.mirror(), 'mate'),
            ('mate_b_white', mirror, 'mate'), ('mate_b_black', mirror.mirror(), 'mate')]


I verified my frozen protocol and protected earlier response/model files.


## 1. I freeze the sample and split

I exclude all earlier exposed/reserved IDs and solver positions, including the previous mixed-length run. I use the board after Lichess's setup move and require exactly one remaining solver move.

In [3]:
def position_key(fen): return ' '.join(str(fen).split()[:4])
def historical_exclusions():
    ids, positions, sources = set(), set(), []
    def add_id(value):
        if isinstance(value, str) and re.fullmatch('[A-Za-z0-9]{5}', value): ids.add(value)
    def scan_metadata(value):
        if isinstance(value, dict):
            for k, v in value.items():
                if k in ['excluded_ids', 'reserved_ids', 'sample_ids', 'puzzle_ids'] and isinstance(v, list):
                    for item in v: add_id(item)
                elif isinstance(v, (dict, list)): scan_metadata(v)
        elif isinstance(value, list):
            for v in value:
                if isinstance(v, (dict, list)): scan_metadata(v)
    for parent in [ROOT/'data/samples', ROOT/'data/experiments']:
        for p in sorted(parent.rglob('*')):
            if not p.is_file() or OUT in p.parents: continue
            before = len(ids)
            if p.suffix == '.csv':
                frame = pd.read_csv(p, dtype=str, keep_default_na=False)
                for column in ['puzzle_id', 'PuzzleId']:
                    if column in frame:
                        for value in frame[column]: add_id(value)
                for column in ['solver_fen']:
                    if column in frame:
                        positions.update(position_key(v) for v in frame[column] if '/' in v)
            elif p.suffix == '.jsonl' and p.name in ['outcomes.jsonl', 'requests.jsonl']:
                for row in records(p):
                    add_id(row.get('puzzle_id'))
                    body = row.get('request', {})
                    text = json.dumps(body)
                    for match in re.findall(r'FEN: ([rnbqkpRNBQKP1-8/]+ [wb] [KQkq-]+ [a-h1-8-]+)', text):
                        positions.add(position_key(match))
            elif p.suffix == '.json' and any(s in p.name for s in ['sampl', 'manifest', 'reserv']):
                scan_metadata(json.loads(p.read_text(encoding='utf-8')))
            else: continue
            if len(ids) != before or p.name in ['sampling.json', 'pilot_sampling_metadata.json']:
                sources.append({'path': p.relative_to(ROOT).as_posix(), 'sha256': sha(p)})
    return ids, positions, sources

sample_file = OUT/'puzzles.csv'
if sample_file.exists():
    sampling = json.loads((OUT/'sampling.json').read_text())
    assert sha(sample_file) == sampling['sample_sha256']
    puzzles = pd.read_csv(sample_file, dtype={'puzzle_id': str})
else:
    excluded, old_positions, exclusion_sources = historical_exclusions()
    save(OUT/'exclusions.json', {'puzzle_ids': sorted(excluded), 'position_keys': sorted(old_positions), 'sources': exclusion_sources})
    rng = np.random.default_rng(CONFIG['sample_seed'])
    cells = {(s['rating'], s['length']): s for s in CONFIG['cell_sizes']}
    picks = {key: pd.DataFrame() for key in cells}
    feature_path = ROOT/'data/validation/full/validated_features.csv'
    source_path = ROOT/'data/lichess_db_puzzle.csv.zst'
    scanned = 0
    for chunk in pd.read_csv(feature_path, chunksize=200_000, dtype={'puzzle_id': str}):
        frame = chunk.loc[chunk.total_solver_moves.eq(1) & ~chunk.puzzle_id.isin(excluded)].copy()
        frame['_position_key'] = frame.solver_fen.map(position_key)
        frame = frame.loc[~frame._position_key.isin(old_positions)].copy()
        frame['rating_stratum'] = np.where(frame.puzzle_rating <= 1157, 'low', np.where(frame.puzzle_rating <= 1700, 'middle', 'high'))
        frame['length_group'] = np.where(frame.total_solver_moves.eq(1), 'one_move', 'multi_move')
        frame['_priority'] = rng.random(len(frame))
        for key, spec in cells.items():
            group = frame.loc[frame.rating_stratum.eq(key[0]) & frame.length_group.eq(key[1])]
            capacity = spec['n']+128
            picks[key] = pd.concat([picks[key], group.nsmallest(capacity, '_priority')], ignore_index=True).nsmallest(capacity, '_priority')
        scanned += len(chunk)
        if scanned % 1_000_000 == 0: print('I scanned', f'{scanned:,}', 'validated positions.', flush=True)
    seen, selected = set(old_positions), []
    for key, spec in cells.items():
        rows = []
        for row in picks[key].sort_values(['_priority', 'puzzle_id']).to_dict('records'):
            if row['_position_key'] in seen: continue
            seen.add(row['_position_key']); rows.append(row)
            if len(rows) == spec['n']: break
        assert len(rows) == spec['n'], 'I need a larger reservoir in a new version; no sample was frozen.'
        selected.append(pd.DataFrame(rows))
    puzzles = pd.concat(selected, ignore_index=True)
    wanted = set(puzzles.puzzle_id); assert len(wanted) == 1000 and not wanted & excluded
    found = []
    for chunk in pd.read_csv(source_path, compression='zstd', chunksize=200_000, dtype=str,
                             keep_default_na=False, usecols=['PuzzleId', 'FEN', 'Moves', 'GameUrl']):
        matched = chunk.loc[chunk.PuzzleId.isin(wanted)]
        if not matched.empty: found.append(matched)
    source = pd.concat(found).rename(columns={'PuzzleId': 'puzzle_id', 'FEN': 'original_fen', 'Moves': 'official_moves', 'GameUrl': 'game_url'})
    puzzles = puzzles.merge(source, on='puzzle_id', validate='one_to_one')
    puzzles['official_first_move'] = puzzles.official_moves.str.split().str[1]
    puzzles['random_success_probability'] = 1/puzzles.legal_moves
    puzzles['split'] = 'train'
    split_rng = np.random.default_rng(CONFIG['split_seed'])
    for key, spec in cells.items():
        indices = puzzles.index[puzzles.rating_stratum.eq(key[0]) & puzzles.length_group.eq(key[1])].to_numpy()
        puzzles.loc[split_rng.permutation(indices)[:spec['test_n']], 'split'] = 'test'
    # I place a rating/length-balanced training prefix first for the operational check.
    order_rng = np.random.default_rng(CONFIG['order_seed'])
    queues = [list(order_rng.permutation(puzzles.index[puzzles.split.eq('train') & puzzles.rating_stratum.eq(k[0]) & puzzles.length_group.eq(k[1])])) for k in cells]
    order = []
    while any(queues):
        for queue in queues:
            if queue: order.append(queue.pop())
    order += list(order_rng.permutation(puzzles.index[puzzles.split.eq('test')]))
    puzzles = puzzles.loc[order].reset_index(drop=True)
    puzzles['collection_order'] = np.arange(1, 1001)
    puzzles['operational_check'] = puzzles.collection_order.le(CONFIG['warmup_n'])
    puzzles['pilot'] = puzzles.collection_order.le(CONFIG['pilot_n'])
    puzzles = puzzles.drop(columns=['_priority', '_position_key'])
    for row in puzzles.itertuples(index=False):
        board = chess.Board(row.original_fen)
        for i, move in enumerate(row.official_moves.split()):
            assert chess.Move.from_uci(move) in board.legal_moves
            board.push_uci(move)
            if i == 0:
                assert board.fen() == row.solver_fen and board.legal_moves.count() == row.legal_moves
        assert chess.Board(row.solver_fen).is_valid()
    puzzles.to_csv(sample_file, index=False)
    sampling = {'n': 1000, 'train_n': 800, 'test_n': 200, 'warmup_n': 20,
                'sample_seed': CONFIG['sample_seed'], 'split_seed': CONFIG['split_seed'], 'order_seed': CONFIG['order_seed'],
                'excluded_id_count': len(excluded), 'excluded_position_count': len(old_positions),
                'sample_sha256': sha(sample_file), 'source_sha256': sha(source_path), 'features_sha256': sha(feature_path),
                'exclusions_sha256': sha(OUT/'exclusions.json'),
                'scope': 'Approximately equal allocation to three rating strata; accuracy describes this balanced sample, not natural puzzle frequencies.'}
    save(OUT/'sampling.json', sampling)
    save(OUT/'environment.json', {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                                'chess': chess.__version__, 'created_utc': datetime.now(timezone.utc).isoformat()})
assert puzzles.total_solver_moves.eq(1).all()
assert len(puzzles) == puzzles.puzzle_id.nunique() == puzzles.solver_fen.map(position_key).nunique() == 1000
assert puzzles.split.value_counts().to_dict() == {'train': 800, 'test': 200}
assert puzzles.loc[puzzles.operational_check, 'split'].eq('train').all()
assert not set(puzzles.puzzle_id) & set(json.loads((OUT/'exclusions.json').read_text())['puzzle_ids'])
assert not set(puzzles.solver_fen.map(position_key)) & set(json.loads((OUT/'exclusions.json').read_text())['position_keys'])
display(puzzles.groupby(['rating_stratum', 'length_group', 'split']).size().unstack('split').reindex(['low','middle','high'], level=0))
print('I froze', len(puzzles), 'distinct puzzle positions and excluded', sampling['excluded_id_count'], 'earlier/reserved IDs.')


,split,test,train
rating_stratum,length_group,,
low,one_move,67,267
middle,one_move,66,267
high,one_move,67,266


I froze 1000 distinct puzzle positions and excluded 1282 earlier/reserved IDs.


## 2. I freeze board characteristics

I use rating, legal choices, piece count, solver-relative material balance, themes and phase. Phase and themes are dataset labels. I freeze encoding using training puzzle features, never success labels. Solution length is constant and is excluded.

In [4]:
# I derive predictors from the solver's board, and freeze encoding without outcomes.
phase_names=['opening','middlegame','endgame']
def phase_of(tags):
    found=[name for name in phase_names if name in str(tags).split()]
    return found[0] if len(found)==1 else 'unknown'
features=puzzles.copy()
features['game_phase']=features.themes.map(phase_of)
features['rating_500']=(features.puzzle_rating-1500)/500
features['legal_moves_10']=features.legal_moves/10
features['piece_count_10']=features.piece_count/10
train=features.loc[features.split.eq('train')]
phase_counts=train.game_phase.value_counts()
common=[p for p in ['opening','middlegame','endgame','unknown'] if phase_counts.get(p,0)>=30]
features['phase_group']=features.game_phase.where(features.game_phase.isin(common),'other')
phase_groups=features.loc[features.split.eq('train'),'phase_group'].value_counts()
baseline=str(phase_groups.index[0])
core=['rating_500','legal_moves_10','piece_count_10','material_balance'];extra=[];theme_counts={}
for tag in CONFIG['theme_candidates']:
    column='theme_'+tag;features[column]=features.themes.fillna('').str.split().map(lambda tags:int(tag in tags))
    count=int(features.loc[features.split.eq('train'),column].sum());theme_counts[tag]=count
    if 30<=count<=770:extra.append(column)
for phase in sorted(phase_groups.index):
    if phase!=baseline:
        column='phase_'+phase;features[column]=features.phase_group.eq(phase).astype(int);extra.append(column)
recipe={'core':core,'full':core+extra,'theme_training_counts':theme_counts,'phase_training_counts':phase_counts.to_dict(),
        'common_phases':common,'phase_baseline':baseline,'frozen_before_responses':True}
feature_path=OUT/'analysis_features.csv';recipe_path=OUT/'feature_recipe.json'
if recipe_path.exists():
    assert json.loads(recipe_path.read_text())==recipe
    existing=pd.read_csv(feature_path,dtype={'puzzle_id':str})
    assert existing.puzzle_id.tolist()==features.puzzle_id.tolist()
    pd.testing.assert_frame_equal(existing[recipe['full']],features[recipe['full']],check_dtype=False)
else:
    features.to_csv(feature_path,index=False);save(recipe_path,recipe)
index=features[['puzzle_id','rating_stratum','split','collection_order','pilot','operational_check']].to_dict('records')
index_path=CFG/'sample_index.json'
if index_path.exists():assert json.loads(index_path.read_text())==index
else:save(index_path,index)
assert len(features)==1000 and len(features.loc[features.pilot])==100
assert features.loc[features.pilot,'split'].eq('train').all()
assert features[recipe['full']].notna().all().all()
setup={'ready_for_pilot':True,'new_api_requests':0,'n':1000,'pilot_n':100,'train_n':800,'test_n':200,
       'sample_sha256':sha(sample_file),'feature_sha256':sha(feature_path),'recipe_sha256':sha(recipe_path),
       'manifest_sha256':sha(OUT/'manifest.json'),'historical_data_files_preserved':len(protected),
       'next_step':'Notebook 04: change RUN_MODE from setup to pilot only after cooldown. No calls have been sent.'}
save(OUT/'preparation.json',setup)
display(pd.DataFrame([{'Predictor set':k,'Columns':', '.join(recipe[k])} for k in ['core','full']]))
print('My fresh sample, pilot prefix, feature encoding and protocol are frozen. No API requests were sent.')


,Predictor set,Columns
0,core,"rating_500, legal_moves_10, piece_count_10, ma..."
1,full,"rating_500, legal_moves_10, piece_count_10, ma..."


My fresh sample, pilot prefix, feature encoding and protocol are frozen. No API requests were sent.


## 3. Why I moved from local to hosted models

I needed enough correct and incorrect answers to study associations. These are earlier feasibility checks, not results for my new study. Different samples and prompt formats prevent a controlled ranking.

In [5]:
evidence=pd.read_csv(CFG/'background_evidence.csv')
evidence['Correct / completed']=evidence.correct.astype(str)+' / '+evidence.completed.astype(str)
display(evidence[['experiment','Correct / completed','scope']])


,experiment,Correct / completed,scope
0,Local full-solution pilot,0 / 225,"Full solution; 3 local models, 75 puzzles each"
1,Local assisted first-move pilot,6 / 144,"First move; 3 local models, 48 puzzles each"
2,Hosted GPT-OSS pilot,21 / 60,First move; 60 puzzles
3,Hosted Gemini pilot,32 / 60,First move; same 60 hosted puzzles


## My next step

I open notebook 04 after cooldown and select `RUN_MODE = "pilot"`. It runs capability checks, verifies a 20-training-case resume, and collects up to 100 shared training cases per model. I review notebook 05 afterward. My full regression and XGBoost analysis is notebook 06, following the final_analysis_plan.json amendment. The pilot has started; I review its current counts and saved quota status in notebook 04 and the root README.